In [1]:
import random
import math
from PIL import Image, ImageDraw
from collections import namedtuple
import numpy as np

In [2]:
class Pincelada:

    #inicializar una pincelada
    def __init__(self, points=[], width=0, start=0, end=0, color=(0,0,0), joint=False, tipo = "line"):
        self.points = points      #array de puntos tipo (x,y), points = [(x_0,y_0), (x_1,y_1)...] 
        self.width = width        #ancho de linea
        self.start = start        #angulo de arco - acepta valores entre 0 y 359
        self.end = end            #angulo de arco - acepta valores entre 0 y 359
        self.color = color        #en formato RGB - (r,g,b)
        self.joint = joint        #bolean true or false
        self.tipo = tipo          #puede ser "bold-line", "line" o "arc"

    # asigna color dado un punto aleatorio en la imagen
    # toma como referencia el color en la imagen original
    def set_Color(self, img_width, RGB_original_array):
        random_x = random.randint(0,img_width-1)
        random_y = random.randint(0,img_width-1)
        
        color_r = int(RGB_original_array[random_y, random_x, 0])
        color_g = int(RGB_original_array[random_y, random_x, 1])
        color_b = int(RGB_original_array[random_y, random_x, 2]) 
        self.color = (color_r, color_g, color_b)

    #crea un angulo inicial para el arco entre 0 y 359 aleatoriamente
    def set_Width(self, max_width):
        if(self.width == 0):
            self.width = random.randint(1,max_width)
    
    #crea un angulo inicial para el arco entre 0 y 359 aleatoriamente
    def set_random_Start(self):
        self.start = random.randint(1,359)

    #determina el angulo en el que termina el arco, entre start y 359 aleatoriamente
    def set_random_End(self):
        self.end = random.randint(self.start,359)
    
    #crea dos puntos aleatorios en la imagen
    #
    def create_two_random_Points(self, max_distance,img_width, img_height):

        # crea un punto aleatorio (x,y)      
        x_0 = random.randint(0,img_width-1)
        y_0 = random.randint(0,img_height -1)

        #crea un segundo punto que este a una distancia menor que max_dist
        r = random.randint(0,max_distance)
        angle = math.radians(random.randint(0,359))
        x_1 = r*math.cos(angle)-x_0
        x_2 = r*math.cos(angle)-y_0
        
        self.points = [ (x_0, y_0), (x_1, y_1)]
        
    # crea pincelada aleatoria del tipo dado: "bold-line","line" o "arc"
    def generate_Pincelada(self, tipo, max_width=10, max_distance = 20):
        
        self.create_two_random_Points()
        self.set_Color()
        self.set_Width(max_width)
        
        if(tipo == "arc"):
            self.set_random_Start()
            self.set_random_End()
            
        if(tipo == "bold-line"):
            self.joint = True        
        

In [3]:
class ImageGenerator:
    
    def _init_(self, imagen ="vanGogh2.jpeg"):
        with Image.open(imagen) as img:
            img_RGB = img.convert("RGB")
            self.img_width = img.width
            self.img_height = img.height
            self.RGB_original_array = np.array(img_RGB)

            self.img_shape = RGB_original_array.shape
            self.img_type = RGB_original_array.dtype
            
        self.img_evolutiva = Image.new("RGB", [self.img_width, self.img_height])
        self.draw_image = ImageDraw.Draw(self.img_evolutiva) 
        
    # pincelada = gen o cromosoma?
    def draw(pincelada, draw_image):
        if(pincelada.tipo == "arc"):
            draw_image.arc(xy=pincelada.points, start=pincelada.start, end=pincelada.end, fill = pincelada.color, width = pincelada.width)
        else:
            draw_image.line(xy=pincelada.points, fill = pincelada.color, width = pincelada.width, joint=pincelada.joint)
        return draw_image

    def generate_img_evol(lista_pinceladas):
        for i in range(len(lista_pinceladas)):
            draw(lista_pinceladas[i])



In [4]:
class AlgoritmoEvolutivo:

    def _init_(self, number_generaciones,number_chromosomes):
        self.number_generaciones = number_generaciones
        self.number_chromosomes = number_chromosomes
        self.hedge_lambda = 0
        self.hedge_array = []
        self.fitness = []
        self.list_chromosomes = []

    def mean_square_error(img_original, img_offspring):
        filas, columnas = np.where(hedge_array == 1)
        max_error_posible = 65025 
        error_contenido = max_error_posible
        porcent_hedge = 0.0
        # si al menos un pixel fue coloreado
        if len(filas) != 0: 
            img_original_signo = img_original.astype(np.int32) #para evitar negativos cuando haces la resta
            img_original_signo = img_original_signo[filas, columnas]
            img_offspring_signo = img_offspring.astype(np.int32)
            img_offspring_signo = img_offspring_signo[filas, columnas]
            error_contenido = (img_original_signo - img_offspring_signo)**2
            error_contenido = np.mean(error_contenido)
            porcent_hedge = len(filas)/((len(hedge_array) * len(hedge_array[0]))) 
        error_contenido = (error_contenido/max_error_posible) + (hedge_lambda * (1 - porcent_hedge))
        #print(error_contenido)
        return error_contenido


    def update_Hedge_Array(self, pincelada, img_width, img_height):
        # no se si esta condicion es necesaria
        if len(hedge_array) != 0:
        #crea una nueva imagen con solo la pincelada nueva 
            img_tmp = Image.new("RGB", [img_width, img_height])
            draw_tmp = ImageDraw.Draw(img_tmp)
            draw_tmp = self.draw(pincelada, draw_temp)
            #la convierte a una matrix de 0's y 1's, 1 en coord donde esta dibujado
            array_temp = np.array(img_tmp)

        #actualiza el hedge array con los nuevos 1's
        filas, columnas = np.where(np.all(array_tmp == 255, axis=-1))
        self.hedge_array[filas, columnas] = 1
        return self.hedge_array

    #NO ESTA LISTO , aún tengo que arreglar esto
    def create_Chromosomes():
        for i in range(number_imagenes):
            list_pincelada = []
            for j in range(number_pincelada):
                pincelada = generate_pincelada(img_color_array)
                list_pincelada.append(pincelada)
        chromosome.append(list_pincelada) 

    #NO ESTA LISTO; aun tengo que arreglar esta funcion
    def calculate_Fitness(self):
        for i in range(self.number_chromosomes):
            img_evol = Image.new("RGB", [img_width, img_height])
            hedge_array = np.zeros((img_height, img_width))
            generate_img(self.list_chromosomes[i], img_evol, self.hedge_array, img_color_array.shape[1], img_color_array.shape[0])
            value = oo_mse(img_color_array, np.array(img_evol), hedge_array)
            fitness.append(value)
            
    #NO ESTA LISTO tambien tengo que arreglar esto
    def show_Best_Option():
        best_option = chromosome[fitness.index(min(fitness))]
        generate_img(best_option, img_evol, [],img_array.shape[1], img_array.shape[0])
        img_evol.show()